# PyRadiomics 入門

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kyorin-phys/MLIntro2026/blob/main/appendix/radiomics_intro.ipynb)

**ラジオミクス (Radiomics)** は、CT・MRI などの医用画像から腫瘍などの関心領域 (ROI) の特徴量を大量に数値化し、診断や予後予測の機械学習に使う手法です。

[PyRadiomics](https://pyradiomics.readthedocs.io/) は、その特徴量を計算する代表的な Python ライブラリです。

必要なものは2つだけです。

| 入力 | 内容 |
|---|---|
| 画像 (image) | CT・MRI などの画素値 |
| マスク (mask) | ROI の画素を 1、それ以外を 0 とした画像 |

出力される主な特徴量クラス

| クラス | 意味 |
|---|---|
| `shape` | ROI の形 (体積・表面積・球形度など) |
| `firstorder` | ROI 内の画素値の分布 (平均・分散・エントロピーなど) |
| `glcm`, `glrlm`, `glszm`, `gldm`, `ngtdm` | テクスチャ (画素値の空間的な並び方) |

> **このノートブックの範囲**：PyRadiomics の使い方と、特徴量を機械学習につなぐ流れを学ぶ教育用の入門です。
> 医学的な解釈や臨床判断は扱いません。注意点は最後の「このノートブックの範囲と注意点」にまとめています。

> **この教材の作り方**：コードと説明文は、すべて生成 AI (Claude、Gemini) との対話で作成しました。作成者はコードを1行も書いていません。
> 作成者が行ったのは、作りたいものを言葉で伝えること、実行して動作を確かめること、分かりにくい点や問題点を指摘して直させることです。
> 医学的な内容は専門家の確認を受けていません。研究に使う際は、最後の「範囲と注意点」を参考に、各自で確認してください。

## 1. インストール

このノートブックは **Google Colab** で実行します。

PyRadiomics を普通に `pip install` すると、古い NumPy などを自分でビルドし始めて失敗します。
そこで、次の3段階でインストールします。

1. NumPy などのライブラリを先に入れておく
2. PyRadiomics 本体だけを `--no-deps` (依存ライブラリを入れない) で入れる
3. PyRadiomics が使う残りの小さなライブラリを入れる

In [ ]:
# 1) 画像・科学計算のライブラリ (Colab に入っているものはすぐ終わる)
!pip install -q numpy scipy pandas matplotlib SimpleITK pydicom pywavelets

# 2) PyRadiomics 本体 (--no-deps で古い NumPy のビルドを防ぐ)
!pip install -q --no-deps "pyradiomics @ git+https://github.com/AIM-Harvard/pyradiomics@v3.1.0"

# 3) PyRadiomics が使う残りのライブラリ
!pip install -q pykwalify six PyYAML

# このノートブックで使うその他のライブラリ
!pip install -q scikit-image scikit-learn plotly ipywidgets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import SimpleITK as sitk
import radiomics
from radiomics import featureextractor

import logging
radiomics.logger.setLevel(logging.ERROR)  # 余計なログを消す
print(radiomics.__version__)

## 2. 人工的な画像で試す

まずは自分で画像とマスクを作って、特徴量の意味を確かめます。

- 画像A：中が **均一** な球
- 画像B：中が **ザラザラ (ノイズ)** な球

同じ形・同じ平均値なので `shape` や平均値は同じになりますが、テクスチャの特徴量には差が出るはずです。

In [ ]:
size = 64
z, y, x = np.mgrid[:size, :size, :size]
r = np.sqrt((x - 32)**2 + (y - 32)**2 + (z - 32)**2)

mask_arr = (r < 15).astype(np.uint8)          # 半径15の球をROIとする

rng = np.random.default_rng(0)
imgA_arr = np.where(mask_arr == 1, 100, 0).astype(np.float32)            # 均一
imgB_arr = imgA_arr + mask_arr * rng.normal(0, 20, mask_arr.shape)        # ノイズ入り

# numpy配列 → SimpleITK画像 (PyRadiomicsはSimpleITK画像を受け取る)
imgA = sitk.GetImageFromArray(imgA_arr)
imgB = sitk.GetImageFromArray(imgB_arr.astype(np.float32))
mask = sitk.GetImageFromArray(mask_arr)

fig, ax = plt.subplots(1, 3, figsize=(10, 3.5))
ax[0].imshow(imgA_arr[32], cmap='gray'); ax[0].set_title('Image A (uniform)')
ax[1].imshow(imgB_arr[32], cmap='gray'); ax[1].set_title('Image B (noisy)')
ax[2].imshow(mask_arr[32], cmap='gray'); ax[2].set_title('Mask')
for a in ax: a.axis('off')
plt.show()

### 3次元で見る

マスクの表面を三角形のポリゴンにして (`marching_cubes`)、plotly で表示します。
**マウスでドラッグすると回転、ホイールで拡大・縮小** できます。

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from skimage.measure import marching_cubes

def mesh3d(mask_arr, spacing=(1, 1, 1), color='orange'):
    # マスクの表面を三角形のポリゴンにする。spacing は (z, y, x) の順 [mm]
    verts, faces, _, _ = marching_cubes(mask_arr, level=0.5, spacing=spacing)
    z, y, x = verts.T
    i, j, k = faces.T
    return go.Mesh3d(x=x, y=y, z=z, i=i, j=j, k=k, color=color, opacity=1.0,
                     lighting=dict(ambient=0.4, diffuse=0.8, specular=0.3))

def show_side_by_side(traces, titles):
    # 3次元のグラフを横に並べて表示する
    n = len(traces)
    fig = make_subplots(rows=1, cols=n, specs=[[{'type': 'scene'}] * n], subplot_titles=titles)
    for col, trace in enumerate(traces, start=1):
        fig.add_trace(trace, row=1, col=col)
    fig.update_scenes(aspectmode='data')
    fig.update_layout(width=500 * n, height=450, margin=dict(l=0, r=0, t=40, b=0))
    fig.show()

def show3d(mask_arr, spacing=(1, 1, 1), color='orange', title=''):
    show_side_by_side([mesh3d(mask_arr, spacing, color)], [title])

show3d(mask_arr, title='Mask (sphere, r=15)')

中身 (画素値) も見たいときは、球を半分に切った断面を3次元表示します。
画像A は断面が一色、画像B はザラザラに見えるはずです。

In [ ]:
def cut_points(img_arr):
    # x > 32 の半分を取り除き、ROI 内の画素を点で表示して断面を見せる
    zz, yy, xx = np.mgrid[:size, :size, :size]
    keep = (mask_arr == 1) & (xx <= 32)
    return go.Scatter3d(x=xx[keep], y=yy[keep], z=zz[keep], mode='markers',
                        marker=dict(size=2, color=img_arr[keep], colorscale='gray', cmin=40, cmax=160))

show_side_by_side([cut_points(imgA_arr), cut_points(imgB_arr)],
                  ['Image A (uniform) - half cut', 'Image B (noisy) - half cut'])

### 特徴量を抽出する

`RadiomicsFeatureExtractor` を作り、`execute(画像, マスク)` を呼ぶだけです。

In [ ]:
extractor = featureextractor.RadiomicsFeatureExtractor()

resultA = extractor.execute(imgA, mask)
resultB = extractor.execute(imgB, mask)

print('特徴量の数:', len(resultA))

In [ ]:
# 'diagnostics_' で始まるものは設定情報なので除き、表にまとめる
def to_series(result):
    return pd.Series({k: float(v) for k, v in result.items() if not k.startswith('diagnostics_')})

df = pd.DataFrame({'A_uniform': to_series(resultA), 'B_noisy': to_series(resultB)})
print('特徴量の数 (diagnostics除く):', len(df))
df.head(20)

### A と B を比べる

代表的な特徴量を並べました。小数3桁にそろえ、B − A の差と、値が変わった (1%以上違う) ものに ◯ を付けています。

- `shape` (形) と `Mean` (平均) は **同じ** → マスクも平均の明るさも同じだから
- `Variance` (分散)・`Entropy`・`glcm` (テクスチャ) は **B の方が大きい** → 中身がザラザラだから

In [ ]:
cols = [
    'original_shape_VoxelVolume',
    'original_shape_Sphericity',
    'original_firstorder_Mean',
    'original_firstorder_Variance',
    'original_firstorder_Entropy',
    'original_glcm_Contrast',
    'original_glcm_JointEntropy',
]
comp = df.loc[cols].copy()
comp.index = comp.index.str.replace('original_', '')   # 名前を短くする
comp['B - A'] = comp['B_noisy'] - comp['A_uniform']
comp['changed'] = np.where(np.isclose(comp['B_noisy'], comp['A_uniform'], rtol=0.01), '', '◯')

# 小数3桁にそろえて表示する
with pd.option_context('display.float_format', '{:.3f}'.format):
    display(comp)

### クラスごとに見る

全特徴量について、A と B で値が変わったものを、クラスごとに数えます。
差がどのクラスに出ているかが分かります。

In [ ]:
cls = df.index.str.split('_').str[1]   # 'original_glcm_Contrast' → 'glcm'
changed = ~np.isclose(df['A_uniform'], df['B_noisy'], rtol=0.01, equal_nan=True)

summary = pd.DataFrame({
    'total': pd.Series(cls).value_counts(),
    'changed': pd.Series(changed).groupby(cls).sum(),
})
summary

## 3. 実際の MRI 画像で試す

PyRadiomics にはテスト用の脳腫瘍 MRI (`brain1`) が用意されています。
`getTestCase` で画像とマスクのファイルをダウンロードできます。

In [ ]:
imagePath, maskPath = radiomics.getTestCase('brain1')
print(imagePath)
print(maskPath)

image = sitk.ReadImage(imagePath)
label = sitk.ReadImage(maskPath)
print('画像サイズ (x, y, z):', image.GetSize())
print('画素間隔 mm  (x, y, z):', image.GetSpacing())

In [ ]:
img_arr = sitk.GetArrayFromImage(image)   # 配列は (z, y, x) の順
lab_arr = sitk.GetArrayFromImage(label)

# 腫瘍の面積が一番大きいスライスを表示
s = lab_arr.sum(axis=(1, 2)).argmax()

fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
ax[0].imshow(img_arr[s], cmap='gray'); ax[0].set_title(f'MRI slice {s}')
ax[1].imshow(img_arr[s], cmap='gray')
ax[1].imshow(np.ma.masked_where(lab_arr[s] == 0, lab_arr[s]), cmap='autumn', alpha=0.5)
ax[1].set_title('MRI + tumor mask')
for a in ax: a.axis('off')
plt.show()

### スライダーでスライスを動かす

スライダーを動かすと、表示するスライス (z方向の断面) が変わります。腫瘍がどの範囲に広がっているか確かめてみましょう。

In [ ]:
import io
import ipywidgets as widgets
from IPython.display import display

def slice_png(z):
    # スライス z の図を描いて、PNG画像のデータとして返す
    fig, ax = plt.subplots(1, 2, figsize=(9, 4.5))
    ax[0].imshow(img_arr[z], cmap='gray')
    ax[0].set_title(f'MRI slice {z}')
    ax[1].imshow(img_arr[z], cmap='gray')
    ax[1].imshow(np.ma.masked_where(lab_arr[z] == 0, lab_arr[z]), cmap='autumn', alpha=0.5)
    ax[1].set_title(f'tumor pixels: {int((lab_arr[z] > 0).sum())}')
    for a in ax: a.axis('off')
    buf = io.BytesIO()
    fig.savefig(buf, format='png', bbox_inches='tight')
    plt.close(fig)
    return buf.getvalue()

slider = widgets.IntSlider(value=s, min=0, max=img_arr.shape[0] - 1, description='slice')
image_view = widgets.Image(value=slice_png(s), format='png')

def on_change(change):
    image_view.value = slice_png(change['new'])   # 表示中の画像を差し替える

slider.observe(on_change, names='value')
display(widgets.VBox([slider, image_view]))

腫瘍のマスクも3次元で表示してみます。画素間隔 (spacing) を渡すと実寸 (mm) の形になります。

In [ ]:
sx, sy, sz = image.GetSpacing()
show3d(lab_arr > 0, spacing=(sz, sy, sx), color='tomato', title='Tumor mask (brain1)')

In [ ]:
extractor = featureextractor.RadiomicsFeatureExtractor()
result = extractor.execute(imagePath, maskPath)   # ファイルパスを直接渡してもよい

features = to_series(result)
features.head(30)

In [ ]:
# クラスごとの特徴量の数
features.index.str.split('_').str[1].value_counts()

## 4. 設定を変える

よく使う設定の例です。

- `normalize`：画素値を平均0・標準偏差1にそろえる (`normalizeScale` 倍する)。**MRI では重要**
  - CT の画素値 (HU) には絶対的な基準があるが、MRI の画素値は装置や撮像条件で大きく変わるため
- `binWidth`：画素値を何刻みでグループ分けしてからテクスチャを計算するか (既定値 25)
- `resampledPixelSpacing`：画素間隔をそろえる (病院・装置ごとに違うため)
- `enableImageTypes`：元画像 (`Original`) 以外に、フィルタ処理した画像からも特徴量を出す
- `disableAllFeatures` / `enableFeatureClassByName`：計算する特徴量クラスを選ぶ

ここでは `normalizeScale=100` で正規化しているので、画像全体の画素値はおよそ -300〜300 の範囲に収まります。`binWidth=25` はこれを 25 刻みで分けるという意味です (ROI の中だけを見ると段階の数はもっと少なくなります)。

In [ ]:
settings = {
    'normalize': True,                    # 画素値を標準化 (MRI では重要)
    'normalizeScale': 100,                # 標準化した値を100倍する
    'binWidth': 25,
    'resampledPixelSpacing': [1, 1, 1],   # 1mm等方にリサンプリング
    'interpolator': sitk.sitkBSpline,
}
extractor2 = featureextractor.RadiomicsFeatureExtractor(**settings)

extractor2.disableAllFeatures()
extractor2.enableFeatureClassByName('firstorder')
extractor2.enableFeatureClassByName('glcm')

# LoG フィルタ (sigma=1, 3mm) をかけた画像からも計算
extractor2.enableImageTypeByName('LoG', customArgs={'sigma': [1.0, 3.0]})

result2 = to_series(extractor2.execute(imagePath, maskPath))
print('特徴量の数:', len(result2))
result2.index.str.split('_').str[0].value_counts()

## 5. 機械学習につなげる

ラジオミクスの実際の流れは次のとおりです。

1. 症例ごとに **画像 + マスク** を用意する
2. PyRadiomics で特徴量を抽出し、**1症例 = 1行** の表 `X` を作る
3. 診断結果などのラベル `y` と合わせて、機械学習で分類する

ここでは「良性」「悪性」の腫瘍を模した人工データ (各30例) で試します。

| | 良性 | 悪性 |
|---|---|---|
| 境界 | なめらか | デコボコ |
| 中身 | 比較的均一 | 不均一 |
| 大きさ | 同じ分布 | 同じ分布 |

実際の腫瘍と同じように、両者の性質には **重なり** を持たせています。

※ この「良性はなめらか・悪性はデコボコ」は、仕組みを見せるために **作った設定** です。
実際の腫瘍の性質を表したものではありません。

In [ ]:
from scipy.ndimage import gaussian_filter

def make_case(malignant, rng, size=48):
    z, y, x = np.mgrid[:size, :size, :size]
    c = size / 2
    r = np.sqrt((x - c)**2 + (y - c)**2 + (z - c)**2)
    R = rng.uniform(8, 14)                       # 大きさは良性・悪性で同じ分布

    # 悪性ほど境界がデコボコで、中身が不均一 (ただし良性と重なりあり)
    if malignant:
        amp = rng.uniform(0.05, 0.20)            # 境界のデコボコの大きさ
        noise_sd = rng.uniform(10, 25)           # 中身の不均一さ
    else:
        amp = rng.uniform(0.00, 0.10)
        noise_sd = rng.uniform(5, 18)

    bump = gaussian_filter(rng.normal(0, 1, (size, size, size)), sigma=4)
    bump /= bump.std()
    mask = (r < R * (1 + amp * bump)).astype(np.uint8)

    texture = gaussian_filter(rng.normal(0, 1, mask.shape), sigma=1)
    texture /= texture.std()
    img = 100 + noise_sd * texture
    img = np.where(mask == 1, img, 0).astype(np.float32)
    return img, mask

rng = np.random.default_rng(42)
labels = np.array([0] * 30 + [1] * 30)          # 0=良性, 1=悪性 (各30例)
cases = [make_case(m, rng) for m in labels]
print('症例数:', len(cases))

良性・悪性の例を1つずつ3次元で見てみます。

In [ ]:
show_side_by_side([mesh3d(cases[0][1], color='lightgreen'), mesh3d(cases[30][1], color='tomato')],
                  ['Benign (case 0)', 'Malignant (case 30)'])

### 特徴量の表 X を作る

全症例から特徴量を抽出します (少し時間がかかります)。
人工画像は画素値の幅が小さいので `binWidth=5` にしています。

In [ ]:
import time
extractor_ml = featureextractor.RadiomicsFeatureExtractor(binWidth=5)

rows = []
t0=time.time()
for i, (img, msk) in enumerate(cases):
    res = extractor_ml.execute(sitk.GetImageFromArray(img), sitk.GetImageFromArray(msk))
    rows.append(to_series(res))
    if (i + 1) % 10 == 0:
        print(f'{i + 1}/{len(cases)} 症例 完了')

X = pd.DataFrame(rows)
y = labels
print('X の形 (症例数, 特徴量数):', X.shape, time.time()-t0)

In [ ]:
X.head()

### 特徴量1つだけで比べる

まずは良性・悪性で値の分布を箱ひげ図で比べます。どの特徴量も重なりがあり、1つだけではきれいに分けられません。

In [ ]:
show_cols = ['original_shape_VoxelVolume', 'original_shape_Sphericity',
             'original_firstorder_Variance', 'original_glszm_SizeZoneNonUniformityNormalized']

fig, ax = plt.subplots(1, len(show_cols), figsize=(14, 3.5))
for a, col in zip(ax, show_cols):
    a.boxplot([X.loc[y == 0, col], X.loc[y == 1, col]])
    a.set_xticks([1, 2], ['benign', 'malignant'])
    a.set_title(col.replace('original_', ''), fontsize=9)
plt.tight_layout()
plt.show()

### 交差検証で分類性能を評価する

100個以上の特徴量を全部使って、ロジスティック回帰とランダムフォレストで分類します。
症例数が少ないので、**5分割交差検証** で性能を評価します。

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

models = {
    'ロジスティック回帰': make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    'ランダムフォレスト': RandomForestClassifier(n_estimators=300, random_state=0),
}
for name, model in models.items():
    acc = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    auc = cross_val_score(model, X, y, cv=cv, scoring='roc_auc')
    print(f'{name}: 正解率 {acc.mean():.3f} ± {acc.std():.3f},  AUC {auc.mean():.3f} ± {auc.std():.3f}')

### どの特徴量が効いているか

ランダムフォレストの特徴量重要度を見ると、形 (`shape`) とテクスチャ (`glszm`, `glcm` など) の両方が上位に来ます。

In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=0).fit(X, y)
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

imp.head(15)[::-1].plot.barh(figsize=(7, 5))
plt.xlabel('importance')
plt.title('Random forest feature importance (top 15)')
plt.tight_layout()
plt.show()

### 確認：ラベルをでたらめにすると？

特徴量が多く症例が少ないと、偶然に分類できてしまうことがあります (過学習)。
ラベルをシャッフルして同じ評価をすると、正解率は 0.5 (当てずっぽう) 前後になるはずです。
こうなれば、上の結果は偶然ではないと言えます。

In [ ]:
rng2 = np.random.default_rng(0)
y_shuffled = rng2.permutation(y)
acc = cross_val_score(RandomForestClassifier(n_estimators=300, random_state=0), X, y_shuffled, cv=cv)
print(f'ラベルをシャッフルした場合: 正解率 {acc.mean():.3f}')

## 6. 自分のデータで試す (任意)

NIfTI (`.nii.gz`) や NRRD (`.nrrd`) 形式の **画像** と **マスク** の2ファイルを、Colab の左のファイル欄 (フォルダのアイコン) からアップロードし、ファイル名を書き換えて実行してください。

### ⚠ 個人情報を取り除いたデータだけを使うこと

Colab にアップロードしたファイルは Google のサーバーに送られます。**患者さんの画像を、個人情報が付いたままアップロードしてはいけません。**

- **DICOM のヘッダー**：氏名・患者ID・生年月日・検査日などが入っている。NIfTI に変換すると多くは消えるが、ファイル名やフォルダ名に残っていないかも確認する
- **頭部の画像**：3次元にすると顔の形が復元でき、本人だと分かることがある。顔の部分を消す処理 (defacing) が必要
- **施設のルール**：取り除いたデータでも、研究に使うには倫理審査や施設の許可が必要なことがある。分からないときは使う前に確認する

練習するだけなら、公開されている匿名化済みのデータ (例：[The Cancer Imaging Archive (TCIA)](https://www.cancerimagingarchive.net/)) を使うのが安全です。

In [ ]:
# my_image = 'image.nii.gz'
# my_mask  = 'mask.nii.gz'
# my_result = to_series(featureextractor.RadiomicsFeatureExtractor().execute(my_image, my_mask))
# my_result

## このノートブックの範囲と注意点

このノートブックは **道具の使い方** の入門です。実際の研究では、次の点に注意が必要です。

| 注意点 | 内容 |
|---|---|
| マスクのばらつき | 囲む人や方法でマスクが変わり、特徴量も変わる。複数人で囲んで、値が安定する特徴量 (ICC が高いもの) を選ぶことが多い |
| 装置・撮像条件 | 装置や撮像条件 (スライス厚・磁場強度など) が違うと特徴量が変わる。リサンプリングや正規化が必要 |
| MRI の画素値 | CT の HU と違い、MRI の画素値には絶対的な基準がない。正規化 (`normalize`) をしないと症例どうしを比べられない |
| 過学習 | 特徴量の数が症例数よりずっと多いことが普通。交差検証だけでなく、**別の施設のデータでの検証 (外部検証)** が必要 |
| 設定の記録 | `binWidth` などの設定で値が変わるので、論文では使った設定をすべて報告する |

### さらに学ぶための資料

- **IBSI** (Image Biomarker Standardisation Initiative)：特徴量の定義と計算方法の国際的な標準化 — https://theibsi.github.io/
- **RQS** (Radiomics Quality Score)：ラジオミクス研究の質を評価する指標 (Lambin et al., *Nat Rev Clin Oncol*, 2017) — https://doi.org/10.1038/nrclinonc.2017.141
- **CLEAR** チェックリスト：ラジオミクス論文で報告すべき項目 (Kocak et al., *Insights Imaging*, 2023) — https://doi.org/10.1186/s13244-023-01415-8
- PyRadiomics のドキュメント (設定の一覧) — https://pyradiomics.readthedocs.io/en/latest/customization.html

## まとめ

- PyRadiomics は **画像 + マスク** から数百の特徴量を自動計算する
- 1症例 = 1行 の表 (DataFrame) にまとめれば、これまでの授業で扱った機械学習 (ロジスティック回帰・ランダムフォレストなど) の入力にそのまま使える
- 実際の研究では、症例数に比べて特徴量が多すぎるので、**特徴量選択** や **交差検証** が重要になる

### 課題
1. 2章の画像Bのノイズの大きさ (`20`) を変えると `original_glcm_Contrast` はどう変わるか調べてみよう。
2. 4章の `binWidth` を 5 や 50 に変えると、テクスチャ特徴量はどう変わるか調べてみよう。
3. 5章の `make_case` で良性・悪性の差 (`amp` や `noise_sd` の範囲) を小さくすると、分類性能はどう変わるか調べてみよう。
4. 5章で `shape` の特徴量だけ、またはテクスチャの特徴量だけを使うと、分類性能はどう変わるか調べてみよう (ヒント：`X.filter(like='shape')`)。